## Sampling Techniques on Imbalanced Dataset


In [2]:
import pandas as pd
import numpy as np

# ML
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier




In [3]:

url = "https://raw.githubusercontent.com/AnjulaMehto/Sampling_Assignment/main/Creditcard_data.csv"
df = pd.read_csv(url)
df.head()


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,1
2,1,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [4]:
X = df.drop('Class', axis=1)
y = df['Class']


In [5]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.utils import resample


In [6]:
models = {
    "M1": LogisticRegression(max_iter=1000),
    "M2": DecisionTreeClassifier(),
    "M3": RandomForestClassifier(),
    "M4": SVC(),
    "M5": KNeighborsClassifier()
}


Sampling 1: Simple Random Sampling

In [7]:
def simple_random_sampling(X, y, frac=0.7):
    data = pd.concat([X, y], axis=1)
    sample = data.sample(frac=frac, random_state=42)
    return sample.drop('Class', axis=1), sample['Class']


Sampling 2: Stratified Sampling

In [8]:
def stratified_sampling(X, y, frac=0.7):
    return train_test_split(
        X, y, test_size=1-frac, stratify=y, random_state=42
    )[:2]


Sampling 3: Cluster Sampling

In [18]:
def cluster_sampling(X, y, n_clusters=5, selected_clusters=3):
    data = pd.concat([X, y], axis=1).reset_index(drop=True)

    # Create cluster labels
    data['cluster_id'] = data.index % n_clusters

    # Randomly select clusters
    chosen_clusters = np.random.choice(
        data['cluster_id'].unique(),
        size=selected_clusters,
        replace=False
    )

    sampled_data = data[data['cluster_id'].isin(chosen_clusters)]

    return sampled_data.drop(['Class', 'cluster_id'], axis=1), sampled_data['Class']



Sampling 4: Bootstrap Sampling

In [9]:
def bootstrap_sampling(X, y):
    X_boot, y_boot = resample(
        X, y, replace=True, n_samples=len(X), random_state=42
    )
    return X_boot, y_boot


Sampling 5: Cross-Validation (no resampling, evaluation-based):Handled directly during model evaluation.

## Apply Sampling + Train Models

In [11]:
results = pd.DataFrame(
    index=models.keys(),
    columns=[
        "Simple Random",
        "Stratified",
        "Cluster",
        "Bootstrap",
        "Cross Validation"
    ]
)


In [12]:
X_srs, y_srs = simple_random_sampling(X, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_srs, y_srs, test_size=0.3, random_state=42
)

for name, model in models.items():
    model.fit(X_train, y_train)
    results.loc[name, "Simple Random"] = round(
        accuracy_score(y_test, model.predict(X_test)) * 100, 2
    )


In [13]:
X_str, X_test, y_str, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

for name, model in models.items():
    model.fit(X_str, y_str)
    results.loc[name, "Stratified"] = round(
        accuracy_score(y_test, model.predict(X_test)) * 100, 2
    )


In [20]:
X_clu, y_clu = cluster_sampling(X, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_clu, y_clu, test_size=0.3, random_state=42
)

for name, model in models.items():
    model.fit(X_train, y_train)
    results.loc[name, "Cluster"] = round(
        accuracy_score(y_test, model.predict(X_test)) * 100, 2
    )



In [21]:
X_boot, y_boot = bootstrap_sampling(X, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_boot, y_boot, test_size=0.3, random_state=42
)

for name, model in models.items():
    model.fit(X_train, y_train)
    results.loc[name, "Bootstrap"] = round(
        accuracy_score(y_test, model.predict(X_test)) * 100, 2
    )


In [22]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')
    results.loc[name, "Cross Validation"] = round(scores.mean() * 100, 2)


In [25]:
results

,Simple Random,Stratified,Cluster,Bootstrap,Cross Validation
M1,99.38,98.71,97.84,97.41,98.7
M2,98.77,98.71,94.96,99.14,98.19
M3,99.38,98.71,97.84,98.71,98.83
M4,99.38,98.71,97.84,97.41,98.83
M5,99.38,98.71,97.84,97.41,98.83


In [24]:
summary = pd.DataFrame({
    "Best Technique": results.astype(float).idxmax(axis=1),
    "Best Accuracy (%)": results.astype(float).max(axis=1)
})

summary


,Best Technique,Best Accuracy (%)
M1,Simple Random,99.38
M2,Bootstrap,99.14
M3,Simple Random,99.38
M4,Simple Random,99.38
M5,Simple Random,99.38
